In [4]:
# Environment setup
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import warnings
warnings.filterwarnings('ignore')

print("✅ Environment configured")

✅ Environment configured


In [5]:
# Core imports
import gc
import time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import psutil
import pandas as pd
# GPU libraries
import cupy as cp

# NVTabular

# ML libraries
import xgboost as xgb
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold

print("✅ All libraries imported successfully")
print(f"XGBoost version: {xgb.__version__}")

✅ All libraries imported successfully
XGBoost version: 3.0.1


In [6]:
import xgboost as xgb, numpy as np
X = np.random.rand(1000, 50).astype('float32'); y = np.random.randint(0,2,1000)
d = xgb.DMatrix(X, label=y)
try:
    xgb.train({'tree_method':'gpu_hist'}, d, num_boost_round=1)
    print('GPU OK')
except Exception as e:
    print(e)  # "not compiled with GPU support"면 CPU 빌드입니다.

[04:52:47] C:\b\abs_d97hy_84m6\croot\xgboost-split_1749630932152\work\src\common\common.h:181: XGBoost version not compiled with GPU support.


In [7]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

TRAIN_PATH = './train.parquet'
TEST_PATH  = './test.parquet'
TARGET = 'clicked'

# 1) 사용할 열 결정(Parquet 스키마에서 'seq' 제외)
pf = pq.ParquetFile(TRAIN_PATH)
cols = [c for c in pf.schema.names if c != 'seq']

train = pd.read_parquet(TRAIN_PATH, columns=cols)

# 2) 범주형/연속형 컬럼 정의
#   - 노트북에 있던 후보를 우선 사용, 실제 존재하는 것만 채택
known_cats = ['gender', 'age_group', 'inventory_id', 'day_of_week', 'hour']
categorical_cols = [c for c in known_cats if c in train.columns]

#   - 추가로 dtype이 object/category인 컬럼을 자동 포함(타깃 제외)
auto_cat = [c for c in train.select_dtypes(include=['object', 'category']).columns if c != TARGET]
for c in auto_cat:
    if c not in categorical_cols:
        categorical_cols.append(c)

#   - 연속형은 타깃/범주형을 제외한 나머지 수치형
continuous_cols = [
    c for c in train.columns
    if c not in categorical_cols + [TARGET] and pd.api.types.is_numeric_dtype(train[c])
]

# 3) 범주형 인코더(Train 기준의 사전) 생성
cat_maps = {}
for c in categorical_cols:
    vals = train[c].astype('string')
    cats = pd.Index(sorted(vals.dropna().unique()))
    cat_maps[c] = {v: i + 1 for i, v in enumerate(cats)}  # 0은 OOV/결측용

def apply_cat_maps(df, cat_cols, maps):
    out = df.copy()
    for c in cat_cols:
        m = maps[c]
        out[c] = out[c].astype('string').map(m).fillna(0).astype('int32')
    return out

# 4) 결측치 처리 및 형 변환
#   - 연속형: 결측 0, float32
train_cont = train[continuous_cols].fillna(0).astype('float32') if continuous_cols else pd.DataFrame(index=train.index)
#   - 범주형: 맵 적용(int32)
train_cat  = apply_cat_maps(train[categorical_cols], categorical_cols, cat_maps) if categorical_cols else pd.DataFrame(index=train.index)

# 5) 피처 결합
X_train_df = pd.concat([train_cont, train_cat], axis=1)
y = train[TARGET].to_numpy().astype('int32')

print(f"X shape: {X_train_df.shape}, y shape: {y.shape}")
print(f"categorical: {len(categorical_cols)}, continuous: {len(continuous_cols)}")

X shape: (10704179, 117), y shape: (10704179,)
categorical: 5, continuous: 112


In [8]:
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score

X_np = X_train_df.to_numpy(dtype=np.float32)

params = {
    'objective': 'binary:logistic',
    'eval_metric': 'aucpr',
    'learning_rate': 0.1,
    'max_depth': 8,
    'min_child_weight': 1.0,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'tree_method': 'hist',    # GPU 있으면 'gpu_hist'로 변경
    'lambda': 1.0
}

n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

oof = np.zeros(len(y), dtype=np.float32)
models = []
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_np, y), 1):
    dtrain = xgb.DMatrix(X_np[tr_idx], label=y[tr_idx])
    dvalid = xgb.DMatrix(X_np[va_idx], label=y[va_idx])

    model = xgb.train(
        params,
        dtrain,
        num_boost_round=2000,
        evals=[(dtrain, 'train'), (dvalid, 'valid')],
        early_stopping_rounds=100,
        verbose_eval=100
    )
    models.append(model)
    oof[va_idx] = model.predict(dvalid, iteration_range=(0, model.best_iteration + 1))
    ap = average_precision_score(y[va_idx], oof[va_idx])
    print(f"Fold {fold}: AP={ap:.5f}")

print(f"OOF AP={average_precision_score(y, oof):.5f}")

# 최종 학습(전체 데이터)
dall = xgb.DMatrix(X_np, label=y)
final_model = xgb.train(
    params,
    dall,
    num_boost_round=int(np.mean([m.best_iteration for m in models])) + 50
)

[0]	train-aucpr:0.06491	valid-aucpr:0.06306
[100]	train-aucpr:0.10477	valid-aucpr:0.08281
[200]	train-aucpr:0.12860	valid-aucpr:0.08406
[300]	train-aucpr:0.14922	valid-aucpr:0.08432
[400]	train-aucpr:0.17185	valid-aucpr:0.08414
[418]	train-aucpr:0.17593	valid-aucpr:0.08414
Fold 1: AP=0.08437
[0]	train-aucpr:0.06534	valid-aucpr:0.06322
[100]	train-aucpr:0.10593	valid-aucpr:0.08205
[200]	train-aucpr:0.12761	valid-aucpr:0.08294
[300]	train-aucpr:0.14897	valid-aucpr:0.08305
[339]	train-aucpr:0.15757	valid-aucpr:0.08304
Fold 2: AP=0.08318
[0]	train-aucpr:0.06502	valid-aucpr:0.06215
[100]	train-aucpr:0.10505	valid-aucpr:0.08144
[200]	train-aucpr:0.12770	valid-aucpr:0.08237
[300]	train-aucpr:0.15019	valid-aucpr:0.08261
[394]	train-aucpr:0.17077	valid-aucpr:0.08242
Fold 3: AP=0.08264
[0]	train-aucpr:0.06488	valid-aucpr:0.06311
[100]	train-aucpr:0.10538	valid-aucpr:0.08346
[200]	train-aucpr:0.12735	valid-aucpr:0.08446
[300]	train-aucpr:0.14979	valid-aucpr:0.08459
[342]	train-aucpr:0.15911	valid

In [9]:
# test.parquet이 있을 때만 실행
import os

if os.path.exists(TEST_PATH):
    # 동일 열 스키마 적용
    pf_t = pq.ParquetFile(TEST_PATH)
    cols_t = [c for c in pf_t.schema.names if c != 'seq']
    test = pd.read_parquet(TEST_PATH, columns=[c for c in cols_t if c in train.columns and c != TARGET])

    # 연속형/범주형 처리
    test_cont = test[continuous_cols].fillna(0).astype('float32') if continuous_cols else pd.DataFrame(index=test.index)
    # 범주형은 train에서 만든 맵으로 OOV=0
    present_cats = [c for c in categorical_cols if c in test.columns]
    test_cat = apply_cat_maps(test[present_cats], present_cats, cat_maps) if present_cats else pd.DataFrame(index=test.index)

    X_test_df = pd.concat([test_cont, test_cat], axis=1)
    dtest = xgb.DMatrix(X_test_df.to_numpy(dtype=np.float32))
    test_pred = final_model.predict(dtest)

    # 제출 저장(형식은 대회 규칙/샘플에 맞춰 컬럼명 수정)
    sub_path = './submission.csv'
    pd.DataFrame({'clicked': test_pred}).to_csv(sub_path, index=False)
    print(f"Saved: {sub_path}")

Saved: ./submission.csv
